# Part 13 (실습) — StateGraph 직접 짜기

> **이 노트북의 목표**
> "초안 → 검토 → (통과 시 발행 / 반려 시 수정 후 재검토)" 그래프를 처음부터 직접 짬. 상태 정의 → 노드 → 엣지 → 조건 분기 → 루프 + 횟수 제한까지. Part 12에서 "에이전트로는 보장 안 됐던" 절차가 그래프로는 보장됨을 확인함.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~12 · 네 신호 중 ①반복 ②분기 ③상태 구현

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai langgraph

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
# ChatGoogleGenerativeAI: Google Gemini 모델을 LangChain에서 사용할 수 있게 감싸주는 클래스
#   → langchain-google-genai 패키지에서 제공
#   → .invoke() / .stream() / .batch() 등 LangChain 공통 메서드를 지원
from langchain_google_genai import ChatGoogleGenerativeAI
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-3-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0.3)
print("준비 완료")

## 1. 상태(State) 정의

그래프가 들고 다니며 갱신할 데이터를 `TypedDict`로 정의함. `history`는 누적해야 하니 리듀서(`operator.add`)를 붙임.

In [ ]:
# typing: 파이썬 타입 힌트 모듈 — 함수/변수의 타입을 명시하여 코드 가독성 향상
#   → TypedDict: 딕셔너리의 키와 값 타입을 정의하는 클래스
#   → Annotated: 타입에 추가 메타데이터(리듀서 등)를 붙이는 데 사용
from typing import TypedDict, Annotated
# operator: 파이썬 내장 모듈 — 연산자를 함수로 쓸 수 있게 해줌
#   → operator.add: 리스트끼리 합치는 리듀서 함수로 자주 사용됨
import operator

class State(TypedDict):
    topic: str                              # 보고서 주제
    draft: str                              # 현재 초안
    review: str                             # 검토 결과
    attempts: int                           # 수정 시도 횟수
    history: Annotated[list, operator.add]  # 흐름 기록 (누적: 리듀서)

print("상태 정의 완료 — history만 누적, 나머지는 덮어쓰기")

## 2. 노드 만들기 — 상태 → 부분 갱신

각 노드는 상태를 받아 "갱신할 키만" 딕셔너리로 돌려줌.

In [ ]:
def draft_node(state: State) -> dict:
    """초안 작성"""
    prompt = f"'{state[\'topic\']}' 주제로 2문장 보고서 초안을 써줘."
    # 수정 단계에서 재진입 시 기존 초안을 보강
    if state.get("draft"):
        prompt = f"다음 초안을 검토의견({state[\'review\']})을 반영해 보강해줘:\n{state[\'draft\']}"
# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
    draft = model.invoke(prompt).content.strip()
    return {"draft": draft, "history": [f"초안 작성(시도 {state[\'attempts\']+1})"]}

def review_node(state: State) -> dict:
    """검토 — 통과/반려 판정"""
    # 간단한 규칙: 초안에 '근거'나 숫자가 있으면 통과 (데모용)
    has_evidence = any(ch.isdigit() for ch in state["draft"]) or "근거" in state["draft"]
    result = "통과" if has_evidence else "반려"
    return {"review": result, "history": [f"검토 → {result}"]}

def revise_node(state: State) -> dict:
    """수정 — 시도 횟수 증가"""
    return {"attempts": state["attempts"] + 1,
            "history": [f"수정 지시(검토의견 반영)"]}

def publish_node(state: State) -> dict:
    """발행"""
    return {"history": ["발행 완료 ✅"]}

print("노드 4개 정의 완료: draft, review, revise, publish")

## 3. 조건 분기 — 라우팅 함수

검토 결과 + 시도 횟수를 보고 다음 노드를 결정함. **횟수 제한(안전장치)**도 여기서 처리.

In [ ]:
def route_after_review(state: State) -> str:
    """검토 후 분기: 통과→발행 / 반려→수정 / 너무 많이 시도→강제 종료"""
    if state["review"] == "통과":
        return "publish"
    if state["attempts"] >= 3:          # 무한 루프 방지 (안전장치)
        return "give_up"
    return "revise"

print("라우팅 함수 정의 — 통과/반려/포기(횟수초과) 3갈래")

## 4. 그래프 조립 — 노드·엣지·분기·루프

`add_node`로 등록, `add_edge`로 고정 연결, `add_conditional_edges`로 분기, 수정→검토로 **루프**.

In [ ]:
# ─── LangGraph 핵심 클래스 ───
# StateGraph: 노드(작업)와 엣지(연결)로 이루어진 상태 그래프를 직접 설계하는 클래스
#   → 노드: 각 단계에서 실행할 함수
#   → 엣지: 노드 간의 실행 순서 (조건부 분기 가능)
# START, END: 그래프의 시작점과 끝점을 나타내는 특수 상수
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)

# 노드 등록
# .add_node(): 그래프에 새로운 노드(작업 단계)를 추가
#   → (노드이름, 실행할함수) 형태로 등록
builder.add_node("draft", draft_node)
builder.add_node("review", review_node)
builder.add_node("revise", revise_node)
builder.add_node("publish", publish_node)
builder.add_node("give_up", lambda s: {"history": ["3회 초과 — 검토 중단"]})

# 엣지
# .add_edge(): 두 노드를 순차적으로 연결 (A → B)
builder.add_edge(START, "draft")        # 시작 → 초안
builder.add_edge("draft", "review")     # 초안 → 검토 (항상)
# .add_conditional_edges(): 조건에 따라 다른 노드로 분기하는 엣지 추가
#   → 라우팅 함수의 반환값에 따라 다음 노드가 결정됨
builder.add_conditional_edges(          # 검토 → 분기
    "review", route_after_review,
    {"publish": "publish", "revise": "revise", "give_up": "give_up"}
)
# .add_edge(): 두 노드를 순차적으로 연결 (A → B)
builder.add_edge("revise", "draft")     # 수정 → 다시 초안 작성 (루프!)
builder.add_edge("publish", END)
builder.add_edge("give_up", END)

# .compile(): 그래프를 실행 가능한 형태로 컴파일 (체크포인터 연결 등)
graph = builder.compile()               # 실행 가능한 그래프로
print("그래프 컴파일 완료")

### 그래프 구조 확인
컴파일된 그래프의 구조를 텍스트로 그려봄.

In [ ]:
print(graph.get_graph().draw_ascii())

## 5. 실행 — 흐름이 보장된다

`invoke`로 실행. history를 보면 "초안→검토→(반려→수정→)*통과→발행"의 흐름이 보장됨.

In [ ]:
result = graph.invoke({
    "topic": "생성형 AI 도입 효과",
    "draft": "", "review": "", "attempts": 0, "history": []
})

print("=== 흐름 기록 ===")
for h in result["history"]:
    print(" •", h)
print("\n=== 최종 초안 ===")
print(result["draft"])
print("\n검토 결과:", result["review"], "| 시도:", result["attempts"]+1)

> 📌 검토는 **항상** 거치고, 반려면 **항상** 수정 루프로 가고, 통과해야만 발행됨. Part 12에서 에이전트로는 보장 안 됐던 흐름이, 그래프에선 그림대로 보장됨.

## 6. 단계별로 흐름 관찰 — stream

`stream`으로 노드가 하나씩 실행되는 과정을 실시간으로 봄.

In [ ]:
print("=== 단계별 실행 ===")
# .stream(): 응답을 한 번에 받지 않고 토큰 단위로 실시간 스트리밍
#   → for chunk in llm.stream(...): 형태로 사용, 타이핑 효과 구현
for step in graph.stream({
    "topic": "데이터 분석 자동화", "draft": "", "review": "", "attempts": 0, "history": []
}):
    for node_name, update in step.items():
        hist = update.get("history", [])
        print(f"[{node_name}] {hist}")

## ✏️ 미니 실습

**과제**: `route_after_review`의 횟수 제한을 3 → 2로 바꾸고, 검토 규칙을 더 까다롭게(예: 초안에 '%'가 있어야 통과) 바꿔서, 그래프가 더 빨리 'give_up'으로 가는지 확인하기.

In [ ]:
# TODO: 직접 수정해서 다시 컴파일·실행
# (route_after_review의 >= 3 을 >= 2 로, review_node의 통과 조건을 변경)

## 정리

| 절 | 한 일 | API |
|---|---|---|
| 1 | 상태 정의 | `TypedDict`, `Annotated[list, operator.add]` |
| 2 | 노드 | 상태→부분 갱신 함수 |
| 3 | 분기 라우팅 | 상태 보고 다음 노드 반환 |
| 4 | 조립 | `add_node/add_edge/add_conditional_edges` + 루프 |
| 5~6 | 실행 | `compile()`, `invoke`, `stream` |

### 3줄 요약
1. 상태(TypedDict+리듀서) → 노드(부분 갱신) → 엣지/분기 → compile → invoke.
2. 조건 분기 + 뒤→앞 엣지로 분기와 루프를 직접 구현(횟수 제한 필수).
3. 에이전트로는 보장 안 되던 절차가 그래프에선 그림대로 보장됨.

### ▶ 다음 (Part 14)
네 신호의 마지막 ④사람 개입 — 그래프를 중간에 멈추고(`interrupt`) 사람의 승인을 받아 재개하는 Human-in-the-loop.